# 02 – Baselines: Majority, BM25, TF-IDF + Logistic Regression

**Task:** given a user's question, predict which of the seen intents it belongs to
(then the app shows that intent's answer).

**Why these baselines?**
- **Majority**: the floor. Always guesses the most common intent; any real model must beat it easily.
- **BM25**: classic keyword search used by search engines. It is the natural "no machine learning"
  approach for FAQ-style QA: find the most similar known question by shared words.
- **TF-IDF + Logistic Regression**: a classic *trained* text classifier. If fine-tuned Transformers
  can't beat this, they are not worth their cost.

**Protocol:** fit on `train`, choose settings on `val`, report `test` once at the end.
The `held_out` intents are not used here (they are for out-of-domain tests in Phase 5).

## 0. Setup (works locally and on Google Colab)

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Eddydev-ALU/ML_Technique_I_Summative_sti-qa-assistant.git"

if IN_COLAB:
    # Get the code, install the packages Colab lacks, download the data, rebuild the splits
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank_bm25==0.2.2", "openpyxl==3.1.5"], check=True)
    subprocess.run([sys.executable, "-m", "src.download_data"], check=True)
    subprocess.run([sys.executable, "-m", "src.data_prep"], check=True)
else:
    # Running locally: move to the project root (one level up from notebooks/)
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.baselines import BM25Baseline, MajorityBaseline, TfidfLogRegBaseline
from src.data_prep import load_prepared
from src.evaluate import per_category_accuracy, ranking_metrics, save_result

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

## 1. Load the prepared data

In [ ]:
questions, answer_bank = load_prepared()
train = questions[questions["split"] == "train"].reset_index(drop=True)
val = questions[questions["split"] == "val"].reset_index(drop=True)
test = questions[questions["split"] == "test"].reset_index(drop=True)

print(questions["split"].value_counts().to_string())
print("\nIntents in train:", train["intent_id"].nunique())
print("Val/test intents missing from train:",
      len(set(val["intent_id"]) - set(train["intent_id"])), "/", len(set(test["intent_id"]) - set(train["intent_id"])))

In [ ]:
def evaluate(model, data):
    """Score a dataset with a fitted model and compute ranking metrics."""
    index = {intent: j for j, intent in enumerate(model.intents)}
    true_idx = data["intent_id"].map(index).to_numpy()
    scores = model.score(data["question"].tolist())
    return ranking_metrics(scores, true_idx), scores, true_idx

## 2. Majority baseline

In [ ]:
majority = MajorityBaseline().fit(train["question"], train["intent_id"])
for name, data in [("val", val), ("test", test)]:
    metrics, _, _ = evaluate(majority, data)
    print(name, {k: round(v, 3) if isinstance(v, float) else v for k, v in metrics.items()})

## 3. BM25: choose k1 and b on val
- `k1` controls how fast repeated words stop adding score.
- `b` controls how much longer questions are penalised.

In [ ]:
bm25_grid = []
for k1 in [0.9, 1.2, 1.5, 2.0]:
    for b in [0.5, 0.75, 1.0]:
        model = BM25Baseline(k1=k1, b=b).fit(train["question"], train["intent_id"])
        metrics, _, _ = evaluate(model, val)
        bm25_grid.append({"k1": k1, "b": b, **metrics})

bm25_grid = pd.DataFrame(bm25_grid).sort_values("accuracy@1", ascending=False)
bm25_grid.round(3)

In [ ]:
best_bm25 = bm25_grid.iloc[0]
bm25 = BM25Baseline(k1=best_bm25["k1"], b=best_bm25["b"]).fit(train["question"], train["intent_id"])
print(f"Chosen BM25 settings: k1={best_bm25['k1']}, b={best_bm25['b']}")

## 4. TF-IDF + Logistic Regression: choose features and C on val
- **word (1,2)**: single words and word pairs ("genital sores").
- **char_wb (2,5)**: 2–5 letter pieces inside words, robust to spelling mistakes ("gonorhea").
- **C**: small = simpler model, large = fits training data more closely (risk of overfitting).

In [ ]:
features = {"word_1-2": ("word", (1, 2)), "char_2-5": ("char_wb", (2, 5))}
lr_grid = []
for feat_name, (analyzer, ngrams) in features.items():
    for C in [1.0, 10.0, 100.0, 1000.0]:
        model = TfidfLogRegBaseline(analyzer=analyzer, ngram_range=ngrams, C=C).fit(train["question"], train["intent_id"])
        val_metrics, _, _ = evaluate(model, val)
        train_metrics, _, _ = evaluate(model, train)
        lr_grid.append({"features": feat_name, "C": C, "train_acc@1": train_metrics["accuracy@1"], **val_metrics})

lr_grid = pd.DataFrame(lr_grid).sort_values("accuracy@1", ascending=False)
lr_grid.round(3)

`train_acc@1` vs val `accuracy@1`: a big gap means the model memorises training questions (overfitting).

In [ ]:
best_lr = lr_grid.iloc[0]
analyzer, ngrams = features[best_lr["features"]]
tfidf_lr = TfidfLogRegBaseline(analyzer=analyzer, ngram_range=ngrams, C=best_lr["C"]).fit(train["question"], train["intent_id"])
print(f"Chosen TF-IDF+LR settings: features={best_lr['features']}, C={best_lr['C']}")

## 5. Final results (val and test) and save to results/experiments.csv

In [ ]:
final_models = {
    "B0_majority": (majority, "always predicts the most frequent training intent"),
    "B1_bm25_q2q": (bm25, f"question->question BM25, k1={best_bm25['k1']}, b={best_bm25['b']}"),
    "B2_tfidf_logreg": (tfidf_lr, f"TF-IDF {best_lr['features']}, C={best_lr['C']}"),
}
rows, test_outputs = [], {}
for name, (model, notes) in final_models.items():
    for split_name, data in [("val", val), ("test", test)]:
        metrics, scores, true_idx = evaluate(model, data)
        save_result(name, split_name, metrics, notes)
        rows.append({"experiment": name, "split": split_name, **metrics})
        if split_name == "test":
            test_outputs[name] = (scores, true_idx)

results = pd.DataFrame(rows)
results.round(3)

## 6. Test accuracy per category

In [ ]:
per_cat = pd.concat({
    name: per_category_accuracy(scores, true_idx, test["category"])["accuracy"]
    for name, (scores, true_idx) in test_outputs.items() if name != "B0_majority"
}, axis=1)
per_cat["n_test"] = test["category"].value_counts()
per_cat.to_csv("results/baselines_per_category.csv")
per_cat

In [ ]:
ax = per_cat.drop(columns="n_test").sort_values("B2_tfidf_logreg").plot.barh(figsize=(8, 5))
ax.set_xlabel("test accuracy@1"); ax.set_xlim(0, 1); ax.set_title("Baselines: test accuracy by category")
plt.tight_layout()
plt.savefig("results/figures/baselines_per_category.png", dpi=150)
plt.show()

## 7. Examples: what the baselines get right and wrong
(Shown for TF-IDF + LR on the test set.)

In [ ]:
scores, true_idx = test_outputs["B2_tfidf_logreg"]
intents = np.array(tfidf_lr.intents)
examples = test[["question", "intent_id"]].copy()
examples["predicted"] = intents[scores.argmax(axis=1)]
examples["confidence"] = scores.max(axis=1).round(3)
examples["bm25_predicted"] = np.array(bm25.intents)[test_outputs["B1_bm25_q2q"][0].argmax(axis=1)]
examples["correct"] = examples["predicted"] == examples["intent_id"]

print("Correct:", examples["correct"].sum(), "/", len(examples))
examples[examples["correct"]].sample(8, random_state=SEED)

In [ ]:
examples[~examples["correct"]].sample(12, random_state=SEED)

In [ ]:
# Are wrong predictions at least in the right category? (e.g. HIV testing vs HIV treatment)
wrong = examples[~examples["correct"]]
same_category = wrong["predicted"].str.split("::").str[0] == wrong["intent_id"].str.split("::").str[0]
print(f"Wrong predictions in the correct category: {same_category.mean():.1%} of {len(wrong)}")

In [ ]:
# Confidence of correct vs wrong predictions (useful later for the "I don't know" threshold)
examples.groupby("correct")["confidence"].describe().round(3)

## 8. Observations (fill in after running)
- Which baseline is best, and by how much over majority?
- Which categories are weakest, and why might that be?
- What kinds of mistakes appear (same disease/different intent, keyword overlap, very short questions)?